# APWC coherence lifecycle 実装ノートブック

このノートブックは、テーマバスケットの **coherence lifecycle** を分析し、それを戦略化するための実装です。

前提データは、以下の3つの wide 型 DataFrame です。

- `apwc`: テーマバスケットの APWC 時系列。index は日付、columns はテーマ名。
- `total_ret`: テーマバスケットのトータルリターン系列。index は日付、columns はテーマ名。
- `resid_ret`: テーマバスケットの残差リターン系列。index は日付、columns はテーマ名。

実装する内容は以下です。

| 目的 | 実装 |
|---|---|
| 個別テーマの現在位置を見たい | Coherence RRG |
| テーマが平均的にどう発生・成熟・減衰するか見たい | Event-aligned lifecycle chart |
| Q3/Q4/Q5 の持続性を見たい | State transition matrix |
| Q5 の危険度や exit timing を見たい | Survival / hazard model |
| テーマのライフサイクル型を分類したい | Trajectory clustering |
| 実戦略に落としたい | Lifecycle state machine |

このノートブックでは、論文の Mosaic + bootstrap z-score ではなく、APWC のローリング z-score を実務的な coherence proxy として使います。

## 0. コンセプト

APWC rolling z-score をテーマ coherence の水準と見なし、さらにその変化率を coherence momentum として定義します。


a) Coherence level

\[
L_{t,i} = EMA_{20}(Z_{t,i})
\]

b) Coherence momentum

\[
D_{t,i} = EMA_{10}(Z_{t,i}) - EMA_{40}(Z_{t,i})
\]

ここで、

\[
Z_{t,i} = \frac{APWC_{t,i} - \mu^{(L)}_{t,i}}{\sigma^{(L)}_{t,i}}
\]

です。

戦略の方向は APWC ではなく、残差リターン・モメンタムで決めます。

\[
M^u_{t,i} = \sum_{s=t-K+1}^{t} u_{s,i}
\]

最終的には、lifecycle state ごとの multiplier を残差モメンタム score に掛けます。

\[
score_{t,i} = m(state_{t,i}) \cdot \widetilde{M}^u_{t,i}
\]

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:.6f}".format

## 1. パラメータ

初期状態では synthetic data で end-to-end 実行できます。実データを使う場合は `USE_SAMPLE_DATA=False` にし、3つのパスを指定してください。

In [ ]:
# -----------------------------
# Data settings
# -----------------------------
USE_SAMPLE_DATA = True

APWC_PATH = Path("apwc.csv")
TOTAL_RET_PATH = Path("total_ret.csv")
RESID_RET_PATH = Path("resid_ret.csv")

APWC_SHEET = 0
TOTAL_RET_SHEET = 0
RESID_RET_SHEET = 0
DATE_COL = 0

# -----------------------------
# Coherence settings
# -----------------------------
Z_WINDOW = 252
Z_MIN_PERIODS = 126

LEVEL_EMA_SPAN = 20
MOM_FAST_EMA_SPAN = 10
MOM_SLOW_EMA_SPAN = 40

N_BUCKETS = 5
BUCKET_ON = "z_level"  # "apwc_z" or "z_level"
MOMENTUM_DEADBAND = 0.0

# -----------------------------
# Residual momentum settings
# -----------------------------
MOM_WINDOW = 60
VOL_WINDOW = 60
MOM_SCORE_CLIP = 3.0

# -----------------------------
# Forward / lifecycle settings
# -----------------------------
FORWARD_HORIZON = 60
TRANSITION_HORIZON = 20
EVENT_PRE_DAYS = 60
EVENT_POST_DAYS = 180
EVENT_MIN_GAP = 60

# -----------------------------
# Strategy settings
# -----------------------------
GROSS_EXPOSURE = 1.0
MAX_ABS_WEIGHT = 0.10
MIN_ACTIVE_THEMES = 3
REBALANCE_FREQ = "W-FRI"  # None for daily, "W-FRI" for weekly
TRANSACTION_COST_BPS = 0.0

# Trueなら、週次リバランスでも multiplier=0 の状態に入ったテーマは日次で即時risk-offします。
FORCE_RISK_OFF_DAILY = True

# Lifecycle multipliers.
# Q5 / Crowded state is capped by design, because total-return diagnostics often peak at Q3/Q4.
STATE_MULTIPLIER = {
    "Dormant": 0.0,
    "Forming": 0.0,
    "Early Active": 0.5,
    "Sweet Spot": 1.0,
    "Crowded/Mature": 0.0,
    "Decaying": 0.0,
}

Q5_ALIGNMENT_MULTIPLIER = 0.25
USE_Q5_COMMON_ALIGNMENT = True
DECAYING_MULTIPLIER = 0.0

OUTPUT_DIR = Path("outputs_lifecycle")

## 2. データ読み込み・整合関数

In [ ]:
def read_wide(path: str | Path,
              date_col: int | str = 0,
              sheet_name: int | str = 0) -> pd.DataFrame:
    """CSV / Excel / Parquet の wide 型データを読み込む。"""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path, index_col=date_col, parse_dates=True)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, index_col=date_col, parse_dates=True, sheet_name=sheet_name)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.loc[~df.index.duplicated(keep="last")]
    return df


def validate_wide(df: pd.DataFrame, name: str) -> None:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}.index must be a DatetimeIndex.")
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"{name} has duplicated columns: {dupes[:5]}")
    if df.index.duplicated().any():
        raise ValueError(f"{name} has duplicated dates in index.")


def align_wide(apwc: pd.DataFrame,
               total_ret: pd.DataFrame,
               resid_ret: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """3つの wide DataFrame を共通 index / columns に揃える。"""
    validate_wide(apwc, "apwc")
    validate_wide(total_ret, "total_ret")
    validate_wide(resid_ret, "resid_ret")

    common_idx = apwc.index.intersection(total_ret.index).intersection(resid_ret.index)
    common_cols = apwc.columns.intersection(total_ret.columns).intersection(resid_ret.columns)

    if len(common_idx) == 0:
        raise ValueError("No common dates across apwc, total_ret, and resid_ret.")
    if len(common_cols) == 0:
        raise ValueError("No common theme columns across apwc, total_ret, and resid_ret.")

    return (
        apwc.loc[common_idx, common_cols].sort_index(),
        total_ret.loc[common_idx, common_cols].sort_index(),
        resid_ret.loc[common_idx, common_cols].sort_index(),
    )

## 3. 共通ユーティリティ

In [ ]:
def rolling_z(df: pd.DataFrame,
              window: int,
              min_periods: Optional[int] = None,
              ddof: int = 0) -> pd.DataFrame:
    if min_periods is None:
        min_periods = window
    mu = df.rolling(window, min_periods=min_periods).mean()
    sig = df.rolling(window, min_periods=min_periods).std(ddof=ddof)
    return (df - mu) / sig.replace(0.0, np.nan)


def forward_sum(ret: pd.DataFrame, horizon: int) -> pd.DataFrame:
    """t時点に t+1 から t+horizon までの単純累積リターンを置く。"""
    return ret.shift(-1).iloc[::-1].rolling(horizon, min_periods=horizon).sum().iloc[::-1]


def trailing_sum(ret: pd.DataFrame, window: int) -> pd.DataFrame:
    return ret.rolling(window, min_periods=window).sum()


def slope(y: pd.Series, x: pd.Series) -> float:
    xy = pd.concat([y, x], axis=1).dropna()
    if xy.shape[0] < 2:
        return np.nan
    yv = xy.iloc[:, 0].astype(float)
    xv = xy.iloc[:, 1].astype(float)
    x_dm = xv - xv.mean()
    y_dm = yv - yv.mean()
    denom = float(np.dot(x_dm, x_dm))
    if denom == 0.0:
        return np.nan
    return float(np.dot(x_dm, y_dm) / denom)


def row_qcut_safe(row: pd.Series, n_buckets: int = 5) -> pd.Series:
    """1行を qcut。値が少ない日や同値が多い日にも壊れにくい。"""
    out = pd.Series(np.nan, index=row.index, dtype=float)
    valid = row.dropna()
    if len(valid) < n_buckets:
        return out
    out.loc[valid.index] = pd.qcut(valid.rank(method="first"), q=n_buckets, labels=False) + 1
    return out


def datewise_bucket(df: pd.DataFrame, n_buckets: int = 5) -> pd.DataFrame:
    """各日付の cross-section で n 分位 bucket を作る。1=lowest, n=highest。"""
    return df.apply(row_qcut_safe, axis=1, n_buckets=n_buckets)


def drawdown_series(ret: pd.Series) -> pd.Series:
    ret = ret.dropna()
    if ret.empty:
        return pd.Series(dtype=float)
    wealth = (1.0 + ret).cumprod()
    return wealth / wealth.cummax() - 1.0


def performance_stats(ret: pd.Series, ann_factor: int = 252) -> pd.Series:
    ret = ret.dropna()
    if len(ret) == 0:
        return pd.Series(dtype=float)

    wealth = (1.0 + ret).cumprod()
    dd = wealth / wealth.cummax() - 1.0
    avg = ret.mean()
    vol_daily = ret.std(ddof=0)
    ann_vol = vol_daily * np.sqrt(ann_factor)
    ann_return = wealth.iloc[-1] ** (ann_factor / len(ret)) - 1.0
    downside = ret.where(ret < 0.0, 0.0).std(ddof=0) * np.sqrt(ann_factor)

    return pd.Series({
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": np.nan if vol_daily == 0.0 else avg / vol_daily * np.sqrt(ann_factor),
        "sortino": np.nan if downside == 0.0 else ann_return / downside,
        "max_drawdown": dd.min(),
        "calmar": np.nan if dd.min() == 0.0 else ann_return / abs(dd.min()),
        "daily_hit_rate": (ret > 0.0).mean(),
        "avg_daily_return": avg,
        "daily_vol": vol_daily,
        "skew": ret.skew(),
        "kurtosis": ret.kurtosis(),
        "cumulative_return": wealth.iloc[-1] - 1.0,
        "n_days": len(ret),
    })


def plot_cumulative_returns(returns: Dict[str, pd.Series], title: str) -> None:
    fig, ax = plt.subplots(figsize=(11, 5))
    for label, ret in returns.items():
        ret = ret.dropna()
        if ret.empty:
            continue
        ((1.0 + ret).cumprod() - 1.0).plot(ax=ax, label=label)
    ax.set_title(title)
    ax.set_ylabel("Cumulative return")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_heatmap(df: pd.DataFrame, title: str, fmt: str = ".2f") -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(df.values, aspect="auto")
    ax.set_xticks(np.arange(df.shape[1]))
    ax.set_yticks(np.arange(df.shape[0]))
    ax.set_xticklabels(df.columns, rotation=45, ha="right")
    ax.set_yticklabels(df.index)
    ax.set_title(title)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            val = df.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, format(val, fmt), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## 4. Synthetic sample data

この synthetic data はノートブックの動作確認用です。実証結果ではありません。

In [ ]:
def make_sample_data(n_days: int = 1_250,
                     n_themes: int = 30,
                     seed: int = 11) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2019-01-01", periods=n_days)
    themes = [f"Theme_{i:02d}" for i in range(1, n_themes + 1)]

    market = rng.normal(0.0002, 0.006, size=(n_days, 1))
    resid = rng.normal(0.0, 0.008, size=(n_days, n_themes))
    total = 0.35 * market + resid + rng.normal(0.0, 0.004, size=(n_days, n_themes))
    apwc = rng.normal(0.0, 0.008, size=(n_days, n_themes))

    for j in range(n_themes):
        n_regimes = rng.integers(1, 4)
        for _ in range(n_regimes):
            start = int(rng.integers(130, max(131, n_days - 220)))
            length = int(rng.integers(70, 180))
            end = min(n_days, start + length)
            direction = rng.choice([-1.0, 1.0])

            ramp = np.linspace(0.015, 0.07, max(1, (end - start) // 2))
            plateau = np.full(max(0, end - start - len(ramp)), 0.06)
            bump = np.r_[ramp, plateau][:end-start]
            apwc[start:end, j] += bump + rng.normal(0.0, 0.010, size=end-start)

            drift = direction * rng.uniform(0.0005, 0.0011)
            resid[start:end, j] += drift
            total[start:end, j] += drift

            # Q5相当の極端coherence後半で common component の逆風を少し入れる
            mature_start = start + int(0.65 * (end - start))
            total[mature_start:end, j] -= direction * rng.uniform(0.0002, 0.0008)

            for t in range(start + 1, end):
                resid[t, j] += 0.12 * resid[t - 1, j]
                total[t, j] += 0.07 * resid[t - 1, j]

    return (
        pd.DataFrame(apwc, index=dates, columns=themes),
        pd.DataFrame(total, index=dates, columns=themes),
        pd.DataFrame(resid, index=dates, columns=themes),
    )

## 5. データ読み込み

In [ ]:
if USE_SAMPLE_DATA:
    apwc, total_ret, resid_ret = make_sample_data()
else:
    apwc = read_wide(APWC_PATH, date_col=DATE_COL, sheet_name=APWC_SHEET)
    total_ret = read_wide(TOTAL_RET_PATH, date_col=DATE_COL, sheet_name=TOTAL_RET_SHEET)
    resid_ret = read_wide(RESID_RET_PATH, date_col=DATE_COL, sheet_name=RESID_RET_SHEET)

apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

print("apwc shape      :", apwc.shape)
print("total_ret shape :", total_ret.shape)
print("resid_ret shape :", resid_ret.shape)
print("date range      :", apwc.index.min().date(), "to", apwc.index.max().date())
print("n themes        :", apwc.shape[1])

display(apwc.head())

## 6. Coherence feature engineering

ここで、以後のすべての分析に使う特徴量を作ります。

- `apwc_z`: APWC rolling z-score
- `z_level`: smoothed coherence level
- `z_momentum`: coherence momentum
- `apwc_q`: datewise coherence quintile
- `mom_resid`: residual momentum
- `mom_score`: residual volatility adjusted momentum score
- `common_ret`: total return minus residual return
- `common_alignment`: residual momentum 方向に common component が味方かどうか
- `state`: lifecycle state

In [ ]:
def classify_lifecycle_state(q: float,
                             z_mom: float,
                             deadband: float = 0.0) -> str:
    if pd.isna(q) or pd.isna(z_mom):
        return "Unknown"
    q = int(q)
    if q <= 2:
        return "Forming" if z_mom > deadband else "Dormant"
    if q == 3:
        return "Early Active" if z_mom > -deadband else "Decaying"
    if q == 4:
        return "Sweet Spot" if z_mom > -deadband else "Decaying"
    if q >= 5:
        return "Crowded/Mature" if z_mom > -deadband else "Decaying"
    return "Unknown"


def compute_coherence_features(apwc: pd.DataFrame,
                               total_ret: pd.DataFrame,
                               resid_ret: pd.DataFrame,
                               z_window: int = 252,
                               z_min_periods: int = 126,
                               level_ema_span: int = 20,
                               mom_fast_span: int = 10,
                               mom_slow_span: int = 40,
                               n_buckets: int = 5,
                               bucket_on: str = "z_level",
                               mom_window: int = 60,
                               vol_window: int = 60,
                               mom_score_clip: float = 3.0,
                               deadband: float = 0.0) -> Dict[str, pd.DataFrame]:
    apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

    apwc_z = rolling_z(apwc, z_window, min_periods=z_min_periods)
    z_level = apwc_z.ewm(span=level_ema_span, min_periods=level_ema_span, adjust=False).mean()
    z_fast = apwc_z.ewm(span=mom_fast_span, min_periods=mom_fast_span, adjust=False).mean()
    z_slow = apwc_z.ewm(span=mom_slow_span, min_periods=mom_slow_span, adjust=False).mean()
    z_momentum = z_fast - z_slow

    bucket_base = z_level if bucket_on == "z_level" else apwc_z
    apwc_q = datewise_bucket(bucket_base, n_buckets=n_buckets)

    mom_resid = resid_ret.rolling(mom_window, min_periods=mom_window).sum()
    resid_vol = resid_ret.rolling(vol_window, min_periods=vol_window).std(ddof=0)
    mom_score = mom_resid / (resid_vol * np.sqrt(mom_window)).replace(0.0, np.nan)
    mom_score = mom_score.clip(lower=-mom_score_clip, upper=mom_score_clip)

    common_ret = total_ret - resid_ret
    mom_common = common_ret.rolling(mom_window, min_periods=mom_window).sum()
    common_alignment = np.sign(mom_resid) * mom_common

    state = pd.DataFrame(index=apwc.index, columns=apwc.columns, dtype=object)
    for col in apwc.columns:
        state[col] = [
            classify_lifecycle_state(q, zm, deadband=deadband)
            for q, zm in zip(apwc_q[col].values, z_momentum[col].values)
        ]

    return {
        "apwc_z": apwc_z,
        "z_level": z_level,
        "z_momentum": z_momentum,
        "apwc_q": apwc_q,
        "mom_resid": mom_resid,
        "resid_vol": resid_vol,
        "mom_score": mom_score,
        "common_ret": common_ret,
        "mom_common": mom_common,
        "common_alignment": common_alignment,
        "state": state,
    }


features = compute_coherence_features(
    apwc=apwc,
    total_ret=total_ret,
    resid_ret=resid_ret,
    z_window=Z_WINDOW,
    z_min_periods=Z_MIN_PERIODS,
    level_ema_span=LEVEL_EMA_SPAN,
    mom_fast_span=MOM_FAST_EMA_SPAN,
    mom_slow_span=MOM_SLOW_EMA_SPAN,
    n_buckets=N_BUCKETS,
    bucket_on=BUCKET_ON,
    mom_window=MOM_WINDOW,
    vol_window=VOL_WINDOW,
    mom_score_clip=MOM_SCORE_CLIP,
    deadband=MOMENTUM_DEADBAND,
)

for k, v in features.items():
    print(k, v.shape)

# Part A. Coherence RRG

横軸に `z_level`、縦軸に `z_momentum` を置きます。

- 右側: coherence level が高い
- 上側: coherence が上昇中
- 右上: expanding / investable coherence
- 右下: high coherence だが weakening / decaying

点の大きさは residual momentum score の絶対値、ラベルは直近時点で注目すべきテーマです。

In [ ]:
def plot_coherence_rrg(features: Dict[str, pd.DataFrame],
                       asof: Optional[pd.Timestamp | str] = None,
                       themes: Optional[Sequence[str]] = None,
                       tail: int = 20,
                       annotate_top: int = 15,
                       min_abs_mom_score: float = 0.0,
                       figsize: Tuple[int, int] = (11, 8)) -> None:
    z_level = features["z_level"]
    z_mom = features["z_momentum"]
    mom_score = features["mom_score"]
    state = features["state"]
    q = features["apwc_q"]

    if asof is None:
        asof = z_level.dropna(how="all").index[-1]
    asof = pd.Timestamp(asof)
    asof = z_level.index[z_level.index.get_indexer([asof], method="ffill")[0]]

    if themes is None:
        snap = pd.DataFrame({
            "z_level": z_level.loc[asof],
            "z_momentum": z_mom.loc[asof],
            "mom_score": mom_score.loc[asof],
            "state": state.loc[asof],
            "q": q.loc[asof],
        }).dropna(subset=["z_level", "z_momentum"])
        snap = snap[snap["mom_score"].abs().fillna(0.0) >= min_abs_mom_score]
        # 注目候補: coherence level, momentum score, Q値を合わせてソート
        snap["rank_score"] = snap["z_level"].rank(pct=True) + snap["mom_score"].abs().rank(pct=True)
        themes = snap.sort_values("rank_score", ascending=False).head(annotate_top).index.tolist()
    else:
        themes = [t for t in themes if t in z_level.columns]

    loc = z_level.index.get_loc(asof)
    start = max(0, loc - tail + 1)
    idx = z_level.index[start:loc + 1]

    fig, ax = plt.subplots(figsize=figsize)

    # 全テーマの現在位置を薄く表示
    all_x = z_level.loc[asof]
    all_y = z_mom.loc[asof]
    ax.scatter(all_x, all_y, s=20, alpha=0.25)

    # 選択テーマの軌跡
    for theme in themes:
        x = z_level.loc[idx, theme]
        y = z_mom.loc[idx, theme]
        if x.dropna().empty or y.dropna().empty:
            continue
        ax.plot(x, y, marker="o", markersize=3, linewidth=1.2, alpha=0.8)
        ax.text(x.iloc[-1], y.iloc[-1], str(theme), fontsize=9)

    # 現在時点の選択テーマをサイズ付きで重ねる
    current = pd.DataFrame({
        "x": z_level.loc[asof, themes],
        "y": z_mom.loc[asof, themes],
        "mom_score": mom_score.loc[asof, themes],
    }).dropna()
    if not current.empty:
        size = 40 + 80 * current["mom_score"].abs().clip(0, MOM_SCORE_CLIP) / max(MOM_SCORE_CLIP, 1e-12)
        ax.scatter(current["x"], current["y"], s=size, alpha=0.8)

    ax.axhline(0.0, linewidth=1)
    ax.axvline(0.0, linewidth=1)
    ax.set_title(f"Coherence RRG as of {asof.date()} | tail={tail} days")
    ax.set_xlabel("Coherence level: EMA(APWC z-score)")
    ax.set_ylabel("Coherence momentum: EMA_fast(APWC z) - EMA_slow(APWC z)")
    ax.grid(True, alpha=0.3)
    plt.show()


plot_coherence_rrg(
    features,
    asof=None,
    tail=30,
    annotate_top=12,
    min_abs_mom_score=0.0,
)

# Part B. Event-aligned lifecycle chart

テーマが coherence 状態に入った日を event day として揃え、平均的に coherence がどのように発生・成熟・減衰するかを確認します。

デフォルトでは、`apwc_q >= 3` に初めて入った日を event とします。

In [ ]:
def find_entry_events(apwc_q: pd.DataFrame,
                      entry_q: int = 3,
                      min_gap: int = 60) -> pd.DataFrame:
    """各テーマが entry_q 以上に入ったイベントを抽出する。"""
    events = []
    for theme in apwc_q.columns:
        q = apwc_q[theme]
        active = q >= entry_q
        prev_active = active.shift(1).fillna(False)
        entries = q.index[active & ~prev_active]

        last_pos = -10**9
        for dt in entries:
            pos = q.index.get_loc(dt)
            if pos - last_pos >= min_gap:
                events.append({"date": dt, "theme": theme, "entry_q": entry_q})
                last_pos = pos

    return pd.DataFrame(events).sort_values(["date", "theme"]).reset_index(drop=True)


def build_event_aligned_lifecycle(features: Dict[str, pd.DataFrame],
                                  total_ret: pd.DataFrame,
                                  resid_ret: pd.DataFrame,
                                  events: pd.DataFrame,
                                  pre_days: int = 60,
                                  post_days: int = 180) -> pd.DataFrame:
    """event dateを0として、coherence / return seriesを相対日付で揃える。"""
    z_level = features["z_level"]
    z_mom = features["z_momentum"]
    apwc_q = features["apwc_q"]
    mom_score = features["mom_score"]
    common_ret = total_ret - resid_ret

    rows = []
    dates = z_level.index

    for event_id, row in events.iterrows():
        dt = pd.Timestamp(row["date"])
        theme = row["theme"]
        if theme not in z_level.columns or dt not in dates:
            continue
        loc = dates.get_loc(dt)
        start = max(0, loc - pre_days)
        end = min(len(dates) - 1, loc + post_days)
        idx = dates[start:end + 1]

        event_sign = np.sign(mom_score.loc[dt, theme])
        if pd.isna(event_sign) or event_sign == 0:
            event_sign = 1.0

        rel_days = np.arange(start - loc, end - loc + 1)
        daily_signed_resid = event_sign * resid_ret.loc[idx, theme]
        daily_signed_total = event_sign * total_ret.loc[idx, theme]
        daily_signed_common = event_sign * common_ret.loc[idx, theme]

        # t=0以降の累積。t<0はイベント前の状態としてNaNにする。
        post_mask = rel_days >= 0
        cum_resid = pd.Series(np.nan, index=idx)
        cum_total = pd.Series(np.nan, index=idx)
        cum_common = pd.Series(np.nan, index=idx)
        cum_resid.iloc[np.where(post_mask)[0]] = daily_signed_resid.iloc[np.where(post_mask)[0]].cumsum().values
        cum_total.iloc[np.where(post_mask)[0]] = daily_signed_total.iloc[np.where(post_mask)[0]].cumsum().values
        cum_common.iloc[np.where(post_mask)[0]] = daily_signed_common.iloc[np.where(post_mask)[0]].cumsum().values

        for rel, d in zip(rel_days, idx):
            rows.append({
                "event_id": event_id,
                "event_date": dt,
                "theme": theme,
                "date": d,
                "rel_day": int(rel),
                "z_level": z_level.loc[d, theme],
                "z_momentum": z_mom.loc[d, theme],
                "apwc_q": apwc_q.loc[d, theme],
                "mom_score": mom_score.loc[d, theme],
                "daily_signed_resid": daily_signed_resid.loc[d],
                "daily_signed_total": daily_signed_total.loc[d],
                "daily_signed_common": daily_signed_common.loc[d],
                "cum_signed_resid": cum_resid.loc[d],
                "cum_signed_total": cum_total.loc[d],
                "cum_signed_common": cum_common.loc[d],
            })

    return pd.DataFrame(rows)


def summarize_event_lifecycle(aligned: pd.DataFrame) -> pd.DataFrame:
    return aligned.groupby("rel_day").agg(
        n_events=("event_id", "nunique"),
        avg_z_level=("z_level", "mean"),
        med_z_level=("z_level", "median"),
        avg_z_momentum=("z_momentum", "mean"),
        avg_q=("apwc_q", "mean"),
        avg_cum_signed_resid=("cum_signed_resid", "mean"),
        avg_cum_signed_total=("cum_signed_total", "mean"),
        avg_cum_signed_common=("cum_signed_common", "mean"),
    )


def plot_event_lifecycle(lifecycle_summary: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(11, 5))
    lifecycle_summary["avg_z_level"].plot(ax=ax, label="avg z_level")
    lifecycle_summary["avg_z_momentum"].plot(ax=ax, label="avg z_momentum")
    ax.axvline(0, linewidth=1)
    ax.axhline(0, linewidth=1)
    ax.set_title("Event-aligned coherence lifecycle")
    ax.set_xlabel("Business days from event")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 5))
    lifecycle_summary["avg_cum_signed_resid"].plot(ax=ax, label="avg cumulative signed residual")
    lifecycle_summary["avg_cum_signed_total"].plot(ax=ax, label="avg cumulative signed total")
    lifecycle_summary["avg_cum_signed_common"].plot(ax=ax, label="avg cumulative signed common")
    ax.axvline(0, linewidth=1)
    ax.axhline(0, linewidth=1)
    ax.set_title("Event-aligned signed return after coherence entry")
    ax.set_xlabel("Business days from event")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


events_q3 = find_entry_events(features["apwc_q"], entry_q=3, min_gap=EVENT_MIN_GAP)
print("Number of Q3 entry events:", len(events_q3))
display(events_q3.head())

aligned_q3 = build_event_aligned_lifecycle(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    events=events_q3,
    pre_days=EVENT_PRE_DAYS,
    post_days=EVENT_POST_DAYS,
)

lifecycle_summary_q3 = summarize_event_lifecycle(aligned_q3)
display(lifecycle_summary_q3.head())
plot_event_lifecycle(lifecycle_summary_q3)

# Part C. State transition matrix

APWC bucket または lifecycle state が、一定期間後にどの状態へ移るかを見ます。

ここではデフォルトで `TRANSITION_HORIZON = 20` 営業日後の遷移を計算します。

In [ ]:
def transition_matrix_from_states(state_df: pd.DataFrame,
                                  horizon: int = 20,
                                  states_order: Optional[Sequence[str]] = None,
                                  normalize: bool = True) -> pd.DataFrame:
    current = state_df.stack().rename("current")
    future = state_df.shift(-horizon).stack().rename("future")
    trans = pd.concat([current, future], axis=1).dropna()
    trans = trans[(trans["current"] != "Unknown") & (trans["future"] != "Unknown")]

    counts = pd.crosstab(trans["current"], trans["future"])

    if states_order is not None:
        counts = counts.reindex(index=states_order, columns=states_order, fill_value=0)

    if normalize:
        return counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    return counts


def transition_matrix_from_buckets(bucket_df: pd.DataFrame,
                                   horizon: int = 20,
                                   n_buckets: int = 5,
                                   normalize: bool = True) -> pd.DataFrame:
    current = bucket_df.stack().rename("current")
    future = bucket_df.shift(-horizon).stack().rename("future")
    trans = pd.concat([current, future], axis=1).dropna()
    trans["current"] = trans["current"].astype(int)
    trans["future"] = trans["future"].astype(int)
    counts = pd.crosstab(trans["current"], trans["future"])
    order = list(range(1, n_buckets + 1))
    counts = counts.reindex(index=order, columns=order, fill_value=0)
    if normalize:
        return counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    return counts


def state_dwell_times(state_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for theme in state_df.columns:
        s = state_df[theme].dropna()
        if s.empty:
            continue
        values = s.values
        dates = s.index
        start = 0
        for i in range(1, len(values) + 1):
            if i == len(values) or values[i] != values[start]:
                rows.append({
                    "theme": theme,
                    "state": values[start],
                    "start_date": dates[start],
                    "end_date": dates[i - 1],
                    "duration": i - start,
                })
                start = i
    return pd.DataFrame(rows)


STATE_ORDER = ["Dormant", "Forming", "Early Active", "Sweet Spot", "Crowded/Mature", "Decaying"]

bucket_trans = transition_matrix_from_buckets(
    features["apwc_q"],
    horizon=TRANSITION_HORIZON,
    n_buckets=N_BUCKETS,
    normalize=True,
)

state_trans = transition_matrix_from_states(
    features["state"],
    horizon=TRANSITION_HORIZON,
    states_order=STATE_ORDER,
    normalize=True,
)

print(f"Bucket transition matrix, horizon={TRANSITION_HORIZON} business days")
display(bucket_trans)
plot_heatmap(bucket_trans, title=f"APWC bucket transition matrix, h={TRANSITION_HORIZON}")

print(f"Lifecycle state transition matrix, horizon={TRANSITION_HORIZON} business days")
display(state_trans)
plot_heatmap(state_trans, title=f"Lifecycle state transition matrix, h={TRANSITION_HORIZON}")

dwell = state_dwell_times(features["state"])
dwell_summary = dwell.groupby("state")["duration"].describe().reindex(STATE_ORDER)
display(dwell_summary)

# Part D. Survival / hazard analysis

ここでは、現在 coherence 状態にあるテーマが、次の `horizon` 日以内に coherence state から脱落する確率を見ます。

デフォルトでは、`apwc_q >= 3` を coherent state と定義します。

目的は、特に Q5 / Crowded 状態で exit risk が高いか、age や coherence drawdown が exit risk を高めるかを見ることです。

In [ ]:
def compute_age_peak_drawdown(active: pd.DataFrame,
                              z_level: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """active episode 内での age、episode-to-date peak、drawdown を計算。"""
    age = pd.DataFrame(np.nan, index=active.index, columns=active.columns)
    peak = pd.DataFrame(np.nan, index=active.index, columns=active.columns)
    drawdown = pd.DataFrame(np.nan, index=active.index, columns=active.columns)

    for theme in active.columns:
        a = active[theme].fillna(False).values.astype(bool)
        z = z_level[theme].values
        start = None
        running_peak = np.nan
        for i, is_active in enumerate(a):
            if is_active:
                if start is None:
                    start = i
                    running_peak = z[i]
                else:
                    if pd.notna(z[i]):
                        running_peak = np.nanmax([running_peak, z[i]])
                age.iat[i, age.columns.get_loc(theme)] = i - start
                peak.iat[i, peak.columns.get_loc(theme)] = running_peak
                drawdown.iat[i, drawdown.columns.get_loc(theme)] = z[i] - running_peak
            else:
                start = None
                running_peak = np.nan

    return age, peak, drawdown


def build_hazard_panel(features: Dict[str, pd.DataFrame],
                       total_ret: pd.DataFrame,
                       resid_ret: pd.DataFrame,
                       horizon: int = 20,
                       coherent_q_min: int = 3) -> pd.DataFrame:
    apwc_q = features["apwc_q"]
    z_level = features["z_level"]
    z_mom = features["z_momentum"]
    mom_score = features["mom_score"]
    common_alignment = features["common_alignment"]
    state = features["state"]

    active = apwc_q >= coherent_q_min

    # 次horizon日以内にactiveでない日があるか。
    future_min_active = active.astype(float).shift(-1).iloc[::-1].rolling(
        horizon, min_periods=horizon
    ).min().iloc[::-1]
    exit_next_h = (future_min_active == 0.0) & active

    age, peak, cdd = compute_age_peak_drawdown(active, z_level)

    fwd_total = forward_sum(total_ret, horizon)
    fwd_resid = forward_sum(resid_ret, horizon)
    mom_sign = np.sign(mom_score)
    signed_fwd_total = mom_sign * fwd_total
    signed_fwd_resid = mom_sign * fwd_resid
    signed_fwd_common = signed_fwd_total - signed_fwd_resid

    panel = pd.concat({
        "active": active.stack(),
        "exit_next_h": exit_next_h.stack(),
        "apwc_q": apwc_q.stack(),
        "z_level": z_level.stack(),
        "z_momentum": z_mom.stack(),
        "coherence_age": age.stack(),
        "coherence_peak": peak.stack(),
        "coherence_drawdown": cdd.stack(),
        "mom_score": mom_score.stack(),
        "common_alignment": common_alignment.stack(),
        "state": state.stack(),
        "signed_fwd_total": signed_fwd_total.stack(),
        "signed_fwd_resid": signed_fwd_resid.stack(),
        "signed_fwd_common": signed_fwd_common.stack(),
    }, axis=1).dropna(subset=["active", "exit_next_h", "apwc_q", "z_level", "z_momentum", "coherence_age"])

    panel.index.names = ["date", "theme"]
    panel = panel[panel["active"].astype(bool)].copy()
    panel["exit_next_h"] = panel["exit_next_h"].astype(int)
    panel["apwc_q"] = panel["apwc_q"].astype(int)
    panel["age_bin"] = pd.cut(
        panel["coherence_age"],
        bins=[-1, 20, 40, 60, 120, 252, np.inf],
        labels=["0-20", "21-40", "41-60", "61-120", "121-252", "253+"]
    )
    return panel


def summarize_hazard_panel(hazard_panel: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    by_q = hazard_panel.groupby("apwc_q").agg(
        obs=("exit_next_h", "size"),
        exit_prob=("exit_next_h", "mean"),
        avg_age=("coherence_age", "mean"),
        avg_z_level=("z_level", "mean"),
        avg_z_momentum=("z_momentum", "mean"),
        avg_drawdown=("coherence_drawdown", "mean"),
        avg_signed_fwd_total=("signed_fwd_total", "mean"),
        avg_signed_fwd_resid=("signed_fwd_resid", "mean"),
        avg_signed_fwd_common=("signed_fwd_common", "mean"),
    )

    by_state = hazard_panel.groupby("state").agg(
        obs=("exit_next_h", "size"),
        exit_prob=("exit_next_h", "mean"),
        avg_age=("coherence_age", "mean"),
        avg_z_level=("z_level", "mean"),
        avg_z_momentum=("z_momentum", "mean"),
        avg_drawdown=("coherence_drawdown", "mean"),
        avg_signed_fwd_total=("signed_fwd_total", "mean"),
        avg_signed_fwd_resid=("signed_fwd_resid", "mean"),
    ).reindex(STATE_ORDER)

    by_age = hazard_panel.groupby("age_bin", observed=False).agg(
        obs=("exit_next_h", "size"),
        exit_prob=("exit_next_h", "mean"),
        avg_z_level=("z_level", "mean"),
        avg_z_momentum=("z_momentum", "mean"),
        avg_drawdown=("coherence_drawdown", "mean"),
        avg_signed_fwd_total=("signed_fwd_total", "mean"),
    )

    return by_q, by_state, by_age


hazard_panel = build_hazard_panel(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    horizon=TRANSITION_HORIZON,
    coherent_q_min=3,
)

hazard_by_q, hazard_by_state, hazard_by_age = summarize_hazard_panel(hazard_panel)

print("Hazard by APWC quintile")
display(hazard_by_q)

print("Hazard by lifecycle state")
display(hazard_by_state)

print("Hazard by coherence age")
display(hazard_by_age)

In [ ]:
def fit_hazard_logit(hazard_panel: pd.DataFrame) -> pd.DataFrame:
    """
    optional: sklearnがあれば logistic regression で exit probability を説明する。
    依存ライブラリがなければ係数表ではなく空DataFrameを返す。
    """
    df = hazard_panel.copy()
    df = df.dropna(subset=[
        "exit_next_h", "z_level", "z_momentum", "coherence_age",
        "coherence_drawdown", "mom_score", "common_alignment", "apwc_q"
    ])

    if df.empty or df["exit_next_h"].nunique() < 2:
        print("Not enough data variation for logistic regression.")
        return pd.DataFrame()

    X_num = df[[
        "z_level", "z_momentum", "coherence_age",
        "coherence_drawdown", "mom_score", "common_alignment", "apwc_q"
    ]].astype(float)
    X_num = (X_num - X_num.mean()) / X_num.std(ddof=0).replace(0.0, np.nan)
    X_state = pd.get_dummies(df["state"], prefix="state", drop_first=True)
    X = pd.concat([X_num, X_state], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    y = df.loc[X.index, "exit_next_h"].astype(int)

    try:
        from sklearn.linear_model import LogisticRegression
        model = LogisticRegression(max_iter=1000, solver="lbfgs")
        model.fit(X, y)
        coef = pd.Series(model.coef_[0], index=X.columns, name="logit_coef")
        out = coef.to_frame()
        out["abs_coef"] = out["logit_coef"].abs()
        return out.sort_values("abs_coef", ascending=False)
    except Exception as exc:
        print("sklearn logistic regression was not available or failed:", exc)
        return pd.DataFrame()


hazard_coef = fit_hazard_logit(hazard_panel)
display(hazard_coef)

# Part E. Trajectory clustering

coherence episode を抽出し、各 episode の形状特徴量でクラスタリングします。

想定する分類は以下です。

- spike-and-fade
- slow-build
- persistent plateau
- false start
- multi-wave に近いもの

ここでは外部依存を抑えるため、まずは episode 特徴量を作り、`sklearn` が使える場合だけ KMeans を実行します。

In [ ]:
def extract_coherence_episodes(features: Dict[str, pd.DataFrame],
                               total_ret: pd.DataFrame,
                               resid_ret: pd.DataFrame,
                               coherent_q_min: int = 3,
                               min_length: int = 10) -> pd.DataFrame:
    apwc_q = features["apwc_q"]
    z_level = features["z_level"]
    z_mom = features["z_momentum"]
    mom_score = features["mom_score"]
    common_ret = total_ret - resid_ret

    rows = []
    dates = apwc_q.index

    for theme in apwc_q.columns:
        active = (apwc_q[theme] >= coherent_q_min).fillna(False).values.astype(bool)
        start = None
        for i in range(len(active) + 1):
            if i < len(active) and active[i] and start is None:
                start = i
            if start is not None and (i == len(active) or not active[i]):
                end = i - 1
                duration = end - start + 1
                if duration >= min_length:
                    idx = dates[start:end + 1]
                    z_path = z_level.loc[idx, theme]
                    mom_path = z_mom.loc[idx, theme]
                    q_path = apwc_q.loc[idx, theme]
                    score_path = mom_score.loc[idx, theme]

                    peak_z = z_path.max()
                    peak_date = z_path.idxmax()
                    peak_pos = idx.get_loc(peak_date)
                    time_to_peak = peak_pos
                    end_z = z_path.iloc[-1]
                    start_z = z_path.iloc[0]
                    post_peak_len = max(duration - time_to_peak - 1, 1)
                    decay_speed = (end_z - peak_z) / post_peak_len

                    start_sign = np.sign(score_path.iloc[0])
                    if pd.isna(start_sign) or start_sign == 0:
                        start_sign = np.sign(score_path.mean())
                    if pd.isna(start_sign) or start_sign == 0:
                        start_sign = 1.0

                    signed_resid_return = start_sign * resid_ret.loc[idx, theme].sum()
                    signed_total_return = start_sign * total_ret.loc[idx, theme].sum()
                    signed_common_return = start_sign * common_ret.loc[idx, theme].sum()

                    # simple peak count: z_levelのローカルピーク数
                    z_vals = z_path.dropna().values
                    n_local_peaks = 0
                    if len(z_vals) >= 3:
                        n_local_peaks = int(((z_vals[1:-1] > z_vals[:-2]) & (z_vals[1:-1] > z_vals[2:])).sum())

                    rows.append({
                        "theme": theme,
                        "start_date": idx[0],
                        "end_date": idx[-1],
                        "duration": duration,
                        "start_z": start_z,
                        "end_z": end_z,
                        "peak_z": peak_z,
                        "mean_z": z_path.mean(),
                        "time_to_peak": time_to_peak,
                        "time_to_peak_ratio": time_to_peak / max(duration - 1, 1),
                        "decay_speed": decay_speed,
                        "mean_z_momentum": mom_path.mean(),
                        "min_z_momentum": mom_path.min(),
                        "max_z_momentum": mom_path.max(),
                        "days_q3": int((q_path == 3).sum()),
                        "days_q4": int((q_path == 4).sum()),
                        "days_q5": int((q_path == 5).sum()),
                        "share_q5": float((q_path == 5).mean()),
                        "n_local_peaks": n_local_peaks,
                        "signed_resid_return": signed_resid_return,
                        "signed_total_return": signed_total_return,
                        "signed_common_return": signed_common_return,
                    })
                start = None

    return pd.DataFrame(rows)


def cluster_episodes(episodes: pd.DataFrame,
                     n_clusters: int = 4) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if episodes.empty:
        return episodes.copy(), pd.DataFrame()

    feature_cols = [
        "duration", "peak_z", "mean_z", "time_to_peak_ratio", "decay_speed",
        "mean_z_momentum", "share_q5", "n_local_peaks", "signed_resid_return", "signed_total_return"
    ]
    X = episodes[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()
    if len(X) < n_clusters:
        out = episodes.copy()
        out["cluster"] = np.nan
        return out, pd.DataFrame()

    X_scaled = (X - X.mean()) / X.std(ddof=0).replace(0.0, np.nan)
    X_scaled = X_scaled.dropna(axis=1)

    out = episodes.copy()
    try:
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=n_clusters, random_state=7, n_init=20)
        labels = km.fit_predict(X_scaled)
        out.loc[X_scaled.index, "cluster"] = labels
    except Exception as exc:
        print("sklearn KMeans was not available or failed. Falling back to rule-based labels:", exc)
        # rule-based fallback
        labels = []
        for _, r in X.iterrows():
            if r["duration"] <= X["duration"].quantile(0.35) and r["peak_z"] >= X["peak_z"].quantile(0.65):
                labels.append("spike-and-fade")
            elif r["duration"] >= X["duration"].quantile(0.65) and r["share_q5"] >= 0.25:
                labels.append("persistent-plateau")
            elif r["time_to_peak_ratio"] >= 0.60:
                labels.append("slow-build")
            else:
                labels.append("false-start/other")
        out.loc[X.index, "cluster"] = labels

    summary = out.groupby("cluster").agg(
        episodes=("theme", "size"),
        avg_duration=("duration", "mean"),
        avg_peak_z=("peak_z", "mean"),
        avg_mean_z=("mean_z", "mean"),
        avg_time_to_peak_ratio=("time_to_peak_ratio", "mean"),
        avg_decay_speed=("decay_speed", "mean"),
        avg_share_q5=("share_q5", "mean"),
        avg_n_local_peaks=("n_local_peaks", "mean"),
        avg_signed_resid_return=("signed_resid_return", "mean"),
        avg_signed_total_return=("signed_total_return", "mean"),
        avg_signed_common_return=("signed_common_return", "mean"),
    )
    return out, summary


episodes = extract_coherence_episodes(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    coherent_q_min=3,
    min_length=10,
)

clustered_episodes, cluster_summary = cluster_episodes(episodes, n_clusters=4)
print("Number of coherence episodes:", len(clustered_episodes))
display(cluster_summary)
display(clustered_episodes.head())

# Part F. Lifecycle state machine strategy

lifecycle state に応じて residual momentum score の multiplier を変える戦略です。

デフォルトの考え方は以下です。

| state | multiplier | 解釈 |
|---|---:|---|
| Dormant | 0.0 | 休眠。取引しない |
| Forming | 0.0 | 形成中。watchlist |
| Early Active | 0.5 | Q3。小さく取る |
| Sweet Spot | 1.0 | Q4。主戦場 |
| Crowded/Mature | 0.0 | Q5。基本は抑制 |
| Decaying | 0.0 | 低下中。exit / reduce |

ただし、Q5 で `common_alignment > 0` の場合は、`Q5_ALIGNMENT_MULTIPLIER` を許可します。

In [ ]:
def normalize_weights(score: pd.DataFrame,
                      gross: float = 1.0,
                      max_abs_weight: Optional[float] = 0.10,
                      min_active: int = 3) -> pd.DataFrame:
    def _norm(row: pd.Series) -> pd.Series:
        row = row.fillna(0.0)
        if (row != 0.0).sum() < min_active:
            return row * 0.0
        denom = row.abs().sum()
        if denom == 0.0:
            return row * 0.0
        w = gross * row / denom
        if max_abs_weight is not None:
            w = w.clip(lower=-max_abs_weight, upper=max_abs_weight)
        return w
    return score.apply(_norm, axis=1)


def apply_rebalance(weights: pd.DataFrame, freq: Optional[str] = None) -> pd.DataFrame:
    if freq is None:
        return weights
    return weights.resample(freq).last().reindex(weights.index).ffill().fillna(0.0)


def state_multiplier_frame(state: pd.DataFrame,
                           common_alignment: pd.DataFrame,
                           base_multiplier: Dict[str, float],
                           q5_alignment_multiplier: float = 0.25,
                           use_q5_alignment: bool = True,
                           decaying_multiplier: float = 0.0) -> pd.DataFrame:
    mult = pd.DataFrame(0.0, index=state.index, columns=state.columns)
    for st, val in base_multiplier.items():
        mult = mult.mask(state == st, val)

    if decaying_multiplier != base_multiplier.get("Decaying", 0.0):
        mult = mult.mask(state == "Decaying", decaying_multiplier)

    if use_q5_alignment:
        mult = mult.mask(
            (state == "Crowded/Mature") & (common_alignment > 0),
            q5_alignment_multiplier,
        )
    return mult.astype(float)


def make_lifecycle_state_strategy(features: Dict[str, pd.DataFrame],
                                  total_ret: pd.DataFrame,
                                  resid_ret: pd.DataFrame,
                                  gross: float = 1.0,
                                  max_abs_weight: Optional[float] = 0.10,
                                  min_active: int = 3,
                                  rebalance_freq: Optional[str] = "W-FRI",
                                  transaction_cost_bps: float = 0.0,
                                  force_risk_off_daily: bool = True,
                                  base_multiplier: Optional[Dict[str, float]] = None,
                                  q5_alignment_multiplier: float = 0.25,
                                  use_q5_alignment: bool = True,
                                  decaying_multiplier: float = 0.0) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, pd.DataFrame]]:
    if base_multiplier is None:
        base_multiplier = STATE_MULTIPLIER

    state = features["state"]
    mom_score = features["mom_score"]
    common_alignment = features["common_alignment"]

    mult = state_multiplier_frame(
        state=state,
        common_alignment=common_alignment,
        base_multiplier=base_multiplier,
        q5_alignment_multiplier=q5_alignment_multiplier,
        use_q5_alignment=use_q5_alignment,
        decaying_multiplier=decaying_multiplier,
    )

    score = mult * mom_score
    weights = normalize_weights(score, gross=gross, max_abs_weight=max_abs_weight, min_active=min_active)
    weights = apply_rebalance(weights, freq=rebalance_freq)

    # 週次リバランスでも、multiplier=0 に入ったテーマは即時risk-offする。
    # 再正規化はしないため、次のリバランスまでgross exposureが下がることがある。
    if force_risk_off_daily:
        weights = weights.where(mult != 0.0, 0.0)

    lagged = weights.shift(1).fillna(0.0)
    pnl_total = (lagged * total_ret).sum(axis=1, min_count=1)
    pnl_resid = (lagged * resid_ret).sum(axis=1, min_count=1)
    pnl_common = pnl_total - pnl_resid

    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (transaction_cost_bps / 10_000.0)

    pnl = pd.DataFrame({
        "total": pnl_total,
        "residual": pnl_resid,
        "common": pnl_common,
        "turnover": turnover,
        "cost": cost,
        "total_net": pnl_total - cost,
        "residual_net": pnl_resid - cost,
        "common_net": pnl_common - cost,
    })

    diagnostics = {
        "state_multiplier": mult,
        "score": score,
        "weights": weights,
    }
    return weights, pnl, diagnostics


def make_plain_residual_momentum_strategy(features: Dict[str, pd.DataFrame],
                                          total_ret: pd.DataFrame,
                                          resid_ret: pd.DataFrame,
                                          gross: float = 1.0,
                                          max_abs_weight: Optional[float] = 0.10,
                                          min_active: int = 3,
                                          rebalance_freq: Optional[str] = "W-FRI",
                                          transaction_cost_bps: float = 0.0) -> Tuple[pd.DataFrame, pd.DataFrame]:
    score = features["mom_score"]
    weights = normalize_weights(score, gross=gross, max_abs_weight=max_abs_weight, min_active=min_active)
    weights = apply_rebalance(weights, freq=rebalance_freq)
    lagged = weights.shift(1).fillna(0.0)
    pnl_total = (lagged * total_ret).sum(axis=1, min_count=1)
    pnl_resid = (lagged * resid_ret).sum(axis=1, min_count=1)
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (transaction_cost_bps / 10_000.0)
    pnl = pd.DataFrame({
        "total": pnl_total,
        "residual": pnl_resid,
        "turnover": turnover,
        "cost": cost,
        "total_net": pnl_total - cost,
        "residual_net": pnl_resid - cost,
    })
    return weights, pnl


def make_bucket_band_strategy(features: Dict[str, pd.DataFrame],
                              total_ret: pd.DataFrame,
                              resid_ret: pd.DataFrame,
                              bucket_multipliers: Dict[int, float],
                              gross: float = 1.0,
                              max_abs_weight: Optional[float] = 0.10,
                              min_active: int = 3,
                              rebalance_freq: Optional[str] = "W-FRI",
                              transaction_cost_bps: float = 0.0,
                              force_risk_off_daily: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    q = features["apwc_q"]
    mult = pd.DataFrame(0.0, index=q.index, columns=q.columns)
    for bucket, val in bucket_multipliers.items():
        mult = mult.mask(q == bucket, val)

    score = mult * features["mom_score"]
    weights = normalize_weights(score, gross=gross, max_abs_weight=max_abs_weight, min_active=min_active)
    weights = apply_rebalance(weights, freq=rebalance_freq)
    if force_risk_off_daily:
        weights = weights.where(mult != 0.0, 0.0)
    lagged = weights.shift(1).fillna(0.0)
    pnl_total = (lagged * total_ret).sum(axis=1, min_count=1)
    pnl_resid = (lagged * resid_ret).sum(axis=1, min_count=1)
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (transaction_cost_bps / 10_000.0)
    pnl = pd.DataFrame({
        "total": pnl_total,
        "residual": pnl_resid,
        "turnover": turnover,
        "cost": cost,
        "total_net": pnl_total - cost,
        "residual_net": pnl_resid - cost,
    })
    return weights, pnl


def compare_pnls(pnls: Dict[str, pd.DataFrame], ret_col: str = "total_net") -> pd.DataFrame:
    return pd.DataFrame({name: performance_stats(pnl[ret_col]) for name, pnl in pnls.items()}).T

In [ ]:
# Lifecycle state machine
weights_lifecycle, pnl_lifecycle, diag_lifecycle = make_lifecycle_state_strategy(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    gross=GROSS_EXPOSURE,
    max_abs_weight=MAX_ABS_WEIGHT,
    min_active=MIN_ACTIVE_THEMES,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    force_risk_off_daily=FORCE_RISK_OFF_DAILY,
    base_multiplier=STATE_MULTIPLIER,
    q5_alignment_multiplier=Q5_ALIGNMENT_MULTIPLIER,
    use_q5_alignment=USE_Q5_COMMON_ALIGNMENT,
    decaying_multiplier=DECAYING_MULTIPLIER,
)

# Benchmarks
weights_plain, pnl_plain = make_plain_residual_momentum_strategy(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    gross=GROSS_EXPOSURE,
    max_abs_weight=MAX_ABS_WEIGHT,
    min_active=MIN_ACTIVE_THEMES,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)

# Q3/Q4 band-pass
weights_band, pnl_band = make_bucket_band_strategy(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    bucket_multipliers={1: 0.0, 2: 0.0, 3: 0.5, 4: 1.0, 5: 0.0},
    gross=GROSS_EXPOSURE,
    max_abs_weight=MAX_ABS_WEIGHT,
    min_active=MIN_ACTIVE_THEMES,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    force_risk_off_daily=FORCE_RISK_OFF_DAILY,
)

# Q3/Q4 + soft Q5
weights_soft_q5, pnl_soft_q5 = make_bucket_band_strategy(
    features=features,
    total_ret=total_ret,
    resid_ret=resid_ret,
    bucket_multipliers={1: 0.0, 2: 0.0, 3: 0.5, 4: 1.0, 5: 0.25},
    gross=GROSS_EXPOSURE,
    max_abs_weight=MAX_ABS_WEIGHT,
    min_active=MIN_ACTIVE_THEMES,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    force_risk_off_daily=FORCE_RISK_OFF_DAILY,
)

pnls = {
    "plain_resid_mom": pnl_plain,
    "band_q3q4": pnl_band,
    "soft_q5": pnl_soft_q5,
    "lifecycle_state_machine": pnl_lifecycle,
}

print("Total return net performance")
display(compare_pnls(pnls, ret_col="total_net"))

print("Residual return net performance")
display(compare_pnls(pnls, ret_col="residual_net"))

plot_cumulative_returns(
    {name: pnl["total_net"] for name, pnl in pnls.items()},
    title="Strategy comparison: total net PnL",
)

plot_cumulative_returns(
    {name: pnl["residual_net"] for name, pnl in pnls.items()},
    title="Strategy comparison: residual net PnL",
)

## Strategy diagnostics

lifecycle state machine が、どの state にどれだけリスクを割り当てているかを確認します。

In [ ]:
def summarize_weight_by_state(weights: pd.DataFrame,
                              state: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for st in STATE_ORDER:
        mask = state == st
        gross_in_state = weights.where(mask, 0.0).abs().sum(axis=1)
        net_in_state = weights.where(mask, 0.0).sum(axis=1)
        rows.append({
            "state": st,
            "avg_gross_weight": gross_in_state.mean(),
            "median_gross_weight": gross_in_state.median(),
            "avg_net_weight": net_in_state.mean(),
            "avg_active_names": ((weights.where(mask, 0.0).abs() > 0).sum(axis=1)).mean(),
        })
    return pd.DataFrame(rows).set_index("state")


weight_state_summary = summarize_weight_by_state(weights_lifecycle, features["state"])
display(weight_state_summary)

fig, ax = plt.subplots(figsize=(11, 4))
weights_lifecycle.abs().sum(axis=1).plot(ax=ax, label="gross")
weights_lifecycle.sum(axis=1).plot(ax=ax, label="net")
ax.set_title("Lifecycle strategy exposure")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Part G. 保存

必要であれば、結果を CSV として保存できます。

In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # features
    for name, df in features.items():
        df.to_csv(OUTPUT_DIR / f"feature_{name}.csv")

    events_q3.to_csv(OUTPUT_DIR / "events_q3_entry.csv", index=False)
    lifecycle_summary_q3.to_csv(OUTPUT_DIR / "event_aligned_lifecycle_q3.csv")
    bucket_trans.to_csv(OUTPUT_DIR / "bucket_transition_matrix.csv")
    state_trans.to_csv(OUTPUT_DIR / "state_transition_matrix.csv")
    dwell.to_csv(OUTPUT_DIR / "state_dwell_times.csv", index=False)
    hazard_panel.to_csv(OUTPUT_DIR / "hazard_panel.csv")
    hazard_by_q.to_csv(OUTPUT_DIR / "hazard_by_q.csv")
    hazard_by_state.to_csv(OUTPUT_DIR / "hazard_by_state.csv")
    hazard_by_age.to_csv(OUTPUT_DIR / "hazard_by_age.csv")
    clustered_episodes.to_csv(OUTPUT_DIR / "clustered_coherence_episodes.csv", index=False)
    cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary.csv")
    weights_lifecycle.to_csv(OUTPUT_DIR / "weights_lifecycle_state_machine.csv")
    pnl_lifecycle.to_csv(OUTPUT_DIR / "pnl_lifecycle_state_machine.csv")

    print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
else:
    print("SAVE_OUTPUTS=False. Set it to True to save all outputs.")

# 実務での読み方

1. **Coherence RRG** で個別テーマが Forming / Expanding / Maturing / Decaying のどこにいるかを見る。
2. **Event-aligned lifecycle** で、平均的にどのタイミングで coherence がピークを打ち、signed total return が弱くなるかを見る。
3. **Transition matrix** で Q3/Q4/Q5 の持続性、Q4→Q5、Q5→Q3/Q2 の落ち方を見る。
4. **Hazard panel** で Q5 の exit risk、coherence age、coherence drawdown、common alignment の影響を見る。
5. **Trajectory clustering** で、persistent plateau と spike-and-fade を分ける。
6. **Lifecycle state machine** で、Q3/Q4 を主戦場にし、Q5 は capped / common-aligned sleeve として扱う。

この設計では、APWC を単調な強度シグナルとして使うのではなく、coherence lifecycle の状態判定に使います。